# Lean-13 : Hommage a Alexandre Grothendieck -- Le langage grothendieckien dans Mathlib 4

**Navigation** : [<< Lean-12 Sensitivity](Lean-12-Sensitivity-Theorem.ipynb) | [Index](README.md) | [Lean-15 Kochen-Specker >>](Lean-15-Kochen-Specker.ipynb)

**Kernel** : Python 3 (Mathlib excerpts shown via `subprocess` -> WSL `lean`)

---

> *On peut tout faire pourvu qu'on prenne le temps de comprendre les choses.* -- A. Grothendieck

## Introduction : pourquoi Grothendieck dans une serie Lean ?

Alexandre Grothendieck (1928-2014) a refonde la geometrie algebrique entre 1958 et 1970 autour de l'IHES (Bures-sur-Yvette), en collaboration avec Jean Dieudonne et un nombre considerable d'eleves. Son langage -- categories, foncteurs, foncteurs derives, sites, faisceaux, schemas, topos -- a transforme l'ensemble des mathematiques. Les milliers de pages des **EGA** (Elements de Geometrie Algebrique) et **SGA** (Seminaire de Geometrie Algebrique du Bois-Marie) sont la trace ecrite de ce programme.

**Cet hommage ne pretend pas formaliser EGA ou SGA.** Le but est plus modeste, mais reel : **montrer comment une partie du langage grothendieckien est deja accessible dans Mathlib 4**. Si vous suivez la serie Lean (Lean-2 a Lean-6, Lean-10 LeanDojo), vous avez vu les fondations : types dependants, propositions, tactiques, Mathlib. On va voir maintenant que ces fondations donnent acces a Grothendieck.

### Ce que vous saurez a la fin

1. Reconnaitre les structures categoriques de Mathlib qui implementent les idees de Grothendieck (cribles, sites, topologies de Grothendieck, faisceaux).
2. Localiser dans Mathlib les definitions de schema (`AlgebraicGeometry.Scheme`), de spectre (`Spec`), du site de Zariski et des proprietes locales de morphismes (etale, lisse, separe).
3. Distinguer ce qui est **deja la** (exploitable pedagogiquement), ce qui est **partiel** (utile avec precautions), et ce qui est **hors-scope Mathlib 4 actuel** (cohomologie etale ell-adique, motifs, six operations, GRR).
4. Lire les enonces des theoremes grothendieckiens dans la syntaxe Lean 4 / Mathlib.

### Prerequis

- Familiarite avec Lean 4 et Mathlib (cf Lean-1 a Lean-6).
- Notions de base de theorie des categories (objet, morphisme, foncteur, transformation naturelle). Aucune connaissance prealable de geometrie algebrique n'est requise pour comprendre les enonces.
- Sympathie pour le projet de **comprendre une chose en la plongeant dans le contexte le plus general qui la rend naturelle** (la phrase est de Grothendieck).

### Duree estimee : 60 minutes

### Note technique sur l'execution

Ce notebook utilise un kernel **Python 3**. Les extraits Mathlib sont obtenus en appelant **`lean`** dans WSL via `subprocess`, avec un `LEAN_PATH` qui pointe vers les `.olean` deja compiles dans `sensitivity_lean/.lake/packages/mathlib/`. Cette astuce evite la duree d'un `lake build` complet a chaque cellule (Mathlib est volumineux). Voir le helper `_run_lean_snippet.sh` dans le meme repertoire.

Pour minimiser le temps total, **toutes les requetes `#check` sont consolidees en une seule invocation Lean** (cellule 'setup'), et les sections suivantes ne font que decouper et afficher les fragments correspondants. Cela divise le cout par ~7 (un seul chargement Mathlib au lieu d'un par section).

Pattern emprunte au notebook Lean-15 (Kochen-Specker, recemment merge) : kernel Python + subprocess WSL + Mathlib cache, plutot que kernel `lean4-wsl` direct.

In [1]:
import subprocess
import textwrap
import re
from pathlib import Path

# Path to the helper script that calls Lean against the sensitivity_lean Mathlib cache
SCRIPT_WIN = Path('_run_lean_snippet.sh').resolve()
SCRIPT_WSL = '/mnt/c/' + str(SCRIPT_WIN).replace('C:\\', '').replace('\\', '/')

def run_lean(snippet: str, timeout_s: int = 300) -> str:
    """Run a Lean snippet against the cached Mathlib (sensitivity_lean .lake).

    Returns combined stdout/stderr (Lean #check output goes to stdout).
    """
    snippet = textwrap.dedent(snippet).strip() + '\n'
    cmd = [
        'wsl', '-d', 'Ubuntu', '--', 'bash', '-c',
        f'cat > /tmp/lean13_snippet.lean << "LEAN_EOF"\n{snippet}LEAN_EOF\nbash {SCRIPT_WSL} /tmp/lean13_snippet.lean'
    ]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        return (r.stdout or '') + ((r.stderr or '') if r.returncode != 0 else '')
    except subprocess.TimeoutExpired:
        return f'TIMEOUT after {timeout_s}s'

# Consolidate ALL #checks in a SINGLE Lean invocation.
# Use plain `--` comments (not `/-- ... -/` which Lean parses as docstring expecting a decl).
# This way Mathlib is loaded only once (~80s) instead of 7x.
ALL_CHECKS = '''
import Mathlib.CategoryTheory.Functor.Basic
import Mathlib.CategoryTheory.Yoneda
import Mathlib.CategoryTheory.Sites.Sieves
import Mathlib.CategoryTheory.Sites.Grothendieck
import Mathlib.Topology.Sheaves.Sheaf
import Mathlib.AlgebraicGeometry.Scheme
import Mathlib.AlgebraicGeometry.Spec
import Mathlib.AlgebraicGeometry.Sites.BigZariski
import Mathlib.AlgebraicGeometry.Morphisms.Etale
import Mathlib.AlgebraicGeometry.Morphisms.Smooth
import Mathlib.AlgebraicGeometry.Morphisms.Separated

-- Section: functor and yoneda
#check @CategoryTheory.Functor
#check @CategoryTheory.yoneda
-- Section: sieves and Grothendieck topologies
#check @CategoryTheory.Sieve
#check @CategoryTheory.GrothendieckTopology
-- Section: sheaves on TopCat
#check @TopCat.Presheaf
#check @TopCat.Sheaf
-- Section: schemes
#check @AlgebraicGeometry.Scheme
#check @AlgebraicGeometry.Spec
-- Section: big Zariski site
#check @AlgebraicGeometry.Scheme.zariskiPretopology
#check @AlgebraicGeometry.Scheme.zariskiTopology
#check @AlgebraicGeometry.Scheme.zariskiTopology_eq
-- Section: morphism properties
#check @AlgebraicGeometry.Etale
#check @AlgebraicGeometry.Smooth
#check @AlgebraicGeometry.IsSeparated
'''

print('Running consolidated Lean check against Mathlib cache (~60-120s)...')
raw = run_lean(ALL_CHECKS, timeout_s=400)

# Split output into individual #check entries.
# Each #check output starts with the binding name (e.g. "@CategoryTheory.yoneda :" or "CategoryTheory.Functor :")
SECTIONS_COUNTS = {
    'functor_yoneda': 2,
    'sieves': 2,
    'sheaves': 2,
    'scheme': 2,
    'bigzariski': 3,
    'morphisms': 3,
}

# Drop empty lines, then group by header line ("@?Identifier ... :")
lines = [ln for ln in raw.splitlines() if ln.strip()]
entries = []
cur = []
for ln in lines:
    if re.match(r'^@?[A-Z][\w.]*\s*:', ln) and cur:
        entries.append('\n'.join(cur))
        cur = [ln]
    else:
        cur.append(ln)
if cur:
    entries.append('\n'.join(cur))

CHECKS = {}
idx = 0
for section, count in SECTIONS_COUNTS.items():
    chunk = entries[idx:idx+count]
    CHECKS[section] = '\n\n'.join(chunk) if chunk else '(no output for section)'
    idx += count

# Keep raw available for debugging if shape changes upstream in Mathlib
CHECKS['_raw'] = raw
CHECKS['_entries_count'] = len(entries)

total_expected = sum(SECTIONS_COUNTS.values())
print(f'Mathlib check done. Parsed {len(entries)} entries (expected {total_expected}).')
if len(entries) != total_expected:
    print('Note: entry count differs from expected. Raw output preserved in CHECKS["_raw"].')
print('Setup ok.')

Running consolidated Lean check against Mathlib cache (~60-120s)...


Mathlib check done. Parsed 14 entries (expected 14).
Setup ok.


## 1. Categories et foncteurs : la fondation

Tout le langage grothendieckien repose sur la theorie des categories. Une **categorie** est un type d'objets muni de morphismes composables avec identites. Un **foncteur** entre deux categories preserve cette structure. Mathlib formalise ces notions dans `Mathlib.CategoryTheory.*`.

Le foncteur le plus important pour Grothendieck est probablement le **plongement de Yoneda** : il identifie chaque objet `c` d'une categorie `C` au foncteur `Hom(-, c)`. Cette identification, en apparence anodine, est le moteur de l'enonce "un schema est un foncteur representable sur la categorie des anneaux" (la definition fonctorielle des schemas, parallele a la definition geometrique).

In [2]:
print(CHECKS['functor_yoneda'])

CategoryTheory.Functor : (C : Type u_3) →
  [CategoryTheory.Category.{u_1, u_3} C] →
    (D : Type u_4) → [CategoryTheory.Category.{u_2, u_4} D] → Type (max u_1 u_2 u_3 u_4)

@CategoryTheory.yoneda : {C : Type u_2} →
  [inst : CategoryTheory.Category.{u_1, u_2} C] → CategoryTheory.Functor C (CategoryTheory.Functor Cᵒᵖ (Type u_1))


### Interpretation : Functor et yoneda

| Symbole Lean | Lecture | Idee grothendieckienne |
|--------------|---------|------------------------|
| `CategoryTheory.Functor C D` | un foncteur de `C` vers `D` | un "changement de point de vue" qui preserve la composition |
| `CategoryTheory.yoneda` | foncteur `C -> (C^op -> Type)` | plonge `C` dans la categorie de ses pre-faisceaux |

Le foncteur de Yoneda permet le **lemme de Yoneda** : il y a une bijection naturelle entre les transformations naturelles `Hom(-, c) -> F` et les elements de `F(c)`. Ce lemme est l'outil de base de tout argument categorique chez Grothendieck. Dans Mathlib, il s'enonce `CategoryTheory.yonedaLemma`.

## 2. Cribles et topologies de Grothendieck

La premiere veritable invention grothendieckienne formalisee dans Mathlib est la **topologie de Grothendieck**. Avant Grothendieck, une topologie sur un espace `X` etait un ensemble d'ouverts. Grothendieck a generalise : une topologie sur une categorie est la donnee, pour chaque objet `X`, d'une collection de **cribles couvrants** -- des sous-objets de Yoneda qui jouent le role des recouvrements ouverts.

Cette generalisation permet d'avoir des "topologies" la ou il n'y a pas d'espace topologique : sur la categorie des schemas, sur celle des anneaux commutatifs, etc. Et donc des **faisceaux** sur ces categories.

Dans Mathlib :
- `CategoryTheory.Sieve X` : un crible sur l'objet `X` (un sous-foncteur de `Hom(-, X)`)
- `CategoryTheory.GrothendieckTopology C` : une topologie de Grothendieck sur la categorie `C`

In [3]:
print(CHECKS['sieves'])

@CategoryTheory.Sieve : {C : Type u_2} → [CategoryTheory.Category.{u_1, u_2} C] → C → Type (max u_2 u_1)

CategoryTheory.GrothendieckTopology : (C : Type u_2) → [CategoryTheory.Category.{u_1, u_2} C] → Type (max u_2 u_1)


### Interpretation : Sieve et GrothendieckTopology

Le type `Sieve : C -> Type` produit, pour chaque objet `X : C`, le type des cribles sur `X`. Une topologie de Grothendieck `J : GrothendieckTopology C` est alors une fonction `J : (X : C) -> Set (Sieve X)` qui designe les cribles "couvrants", soumise a trois axiomes :

1. Le crible maximal est couvrant (axiome d'identite).
2. Stabilite par image inverse (pullback_stable).
3. Stabilite transitive (un crible obtenu en raffinant un couvrant par des couvrants est couvrant).

Mathlib fournit dans le meme fichier les **topologies extremes** : `trivial` (seul le crible maximal couvre), `discrete` (tous les cribles couvrent), `dense` (cribles non vides), `atomic` (axiomatise par des familles couvrantes a un seul morphisme).

**Observation pedagogique** : la definition Lean / Mathlib epouse exactement la definition de SGA 4. Lire la definition Lean, c'est lire SGA 4 dans une syntaxe verifiable.

## 3. Faisceaux sur un espace topologique

Avant les sites, Grothendieck a deja revolutionne la theorie des faisceaux dans son article de Tohoku (1957) : il a place les faisceaux dans le cadre des categories abeliennes et a defini la cohomologie comme un foncteur derive.

Mathlib formalise les faisceaux sur un espace topologique (categorie `TopCat`) avec valeurs dans une categorie `C` quelconque. Un **prefaisceau** est un foncteur `(Opens X)^op -> C`, et un **faisceau** est un prefaisceau verifiant la condition de recollement (egaliseur).

In [4]:
print(CHECKS['sheaves'])

TopCat.Presheaf : (C : Type u_3) → [CategoryTheory.Category.{u_2, u_3} C] → TopCat → Type (max u_3 u_2 u_1)

TopCat.Sheaf : (C : Type u_3) → [CategoryTheory.Category.{u_2, u_3} C] → TopCat → Type (max u_3 u_2 u_1)


### Interpretation : prefaisceaux et faisceaux

Le type `TopCat.Presheaf C X` represente les prefaisceaux sur `X` a valeurs dans `C`. Le type `TopCat.Sheaf C X` ajoute la condition de faisceau (egaliseur sur les recouvrements). 

**Note** : Mathlib a deux presentations equivalentes pour les faisceaux -- l'une via les ouverts d'un espace topologique, l'autre via une topologie de Grothendieck generale. Le pont entre les deux est etabli dans `Mathlib.Topology.Sheaves.Forget` et `Mathlib.CategoryTheory.Sites.Sheaf`. Les deux presentations permettent de redire "un faisceau de groupes abeliens sur `X`", mais la presentation Grothendieck est celle qui se generalise aux schemas, aux sites etales, etc.

Tout ceci est dans Mathlib **aujourd'hui**. C'est le langage de Grothendieck, ecrit dans Lean.

## 4. Schemas : remplacer les varietes par du local-affine

La definition d'un **schema** est l'invention centrale d'EGA I (1960). Avant Grothendieck, on faisait de la geometrie algebrique sur des **varietes** definies par des equations polynomiales sur un corps. Grothendieck remplace les varietes par des **espaces localement anneles** dont chaque ouvert est localement de la forme `Spec R` pour un anneau commutatif `R`.

Cette generalisation autorise :
- des **points generiques** (lies aux ideaux premiers non maximaux)
- des coefficients dans n'importe quel anneau (pas seulement un corps algebriquement clos)
- la **theorie arithmetique** (`Spec Z` est un objet legitime, et la geometrie sur lui = theorie des nombres)

Mathlib formalise cette construction :

In [5]:
print(CHECKS['scheme'])

AlgebraicGeometry.Scheme : Type (u_1 + 1)

AlgebraicGeometry.Spec : CommRingCat → AlgebraicGeometry.Scheme


### Interpretation : Scheme et Spec

`AlgebraicGeometry.Scheme : Type (u+1)` est le type des schemas. `AlgebraicGeometry.Spec : CommRingCat -> Scheme` est le foncteur spectre. Concretement, pour un anneau commutatif `R`, `Spec R` est le schema dont :

- l'espace topologique sous-jacent est l'ensemble des **ideaux premiers** de `R`, muni de la topologie de Zariski (les fermes sont les `V(I) = {p : I ⊆ p}` pour `I` ideal)
- le faisceau structural attache a `Spec R` est determine par `R` lui-meme (localisations)

Cette definition est **vraiment** la definition de SGA 1. Pas une approximation, pas un cas particulier : c'est la meme.

**Realite Mathlib 4 actuelle** : la theorie des schemas dans Mathlib est en developpement actif. Les definitions sont stables, beaucoup de proprietes elementaires sont prouvees (separation, finitude, dimension dans certains cas), mais on est **loin** d'EGA IV. C'est pedagogiquement utile, ce n'est pas une formalisation complete d'EGA.

## 5. Site de Zariski : la topologie de Grothendieck sur la categorie des schemas

La **topologie de Zariski sur la categorie des schemas** est l'exemple le plus naturel de topologie de Grothendieck "non spatiale". Elle est definie par une **pretopologie** : une famille de morphismes `{f_i : U_i -> X}` est couvrante si les `f_i` sont des **immersions ouvertes** et `X` est leur union ensembliste.

Mathlib fournit cette topologie dans `Mathlib.AlgebraicGeometry.Sites.BigZariski`. Le nom "gros site" (big site) signifie qu'on prend tous les schemas, pas seulement les ouverts d'un schema fixe.

In [6]:
print(CHECKS['bigzariski'])

AlgebraicGeometry.Scheme.zariskiPretopology : CategoryTheory.Pretopology AlgebraicGeometry.Scheme

AlgebraicGeometry.Scheme.zariskiTopology : CategoryTheory.GrothendieckTopology AlgebraicGeometry.Scheme

AlgebraicGeometry.Scheme.zariskiTopology_eq : AlgebraicGeometry.Scheme.zariskiTopology =
  AlgebraicGeometry.Scheme.zariskiPretopology.toGrothendieck


### Interpretation : le site de Zariski dans Lean

Trois identifiants Mathlib resument tout :

| Nom | Type | Signification |
|-----|------|---------------|
| `Scheme.zariskiPretopology` | `Pretopology Scheme` | la pretopologie : familles d'immersions ouvertes recouvrantes |
| `Scheme.zariskiTopology` | `GrothendieckTopology Scheme` | la topologie de Grothendieck engendree |
| `Scheme.zariskiTopology_eq` | egalite | atteste que la topologie est bien celle engendree par la pretopologie |

Concretement, `zariskiTopology = zariskiPretopology.toGrothendieck`. C'est le lemme `zariskiTopology_eq`. La pretopologie est plus elementaire (definition directe), la topologie de Grothendieck est plus structuree (axiomes de fermeture). Les deux sont equivalentes ici.

**Au passage** : Mathlib a aussi `Scheme.zariskiTopology.Subcanonical`, qui exprime que tous les representables `Hom(-, X)` sont des faisceaux pour cette topologie -- propriete fondamentale qui dit que les schemas eux-memes "se recollent" pour la topologie de Zariski. C'est une consequence non triviale du lemme de Yoneda + recollement.

## 6. Proprietes locales de morphismes : etale, lisse, separe

Une autre tour de force de Grothendieck (et de son ecole) est la classification des **proprietes locales des morphismes de schemas** : etale, lisse, plat, non ramifie, separe, propre, projectif, etc. Chacune capture une nuance d'"etre regulier" et chacune correspond a une notion classique en geometrie complexe ou en arithmetique.

Mathlib formalise plusieurs de ces proprietes dans `Mathlib.AlgebraicGeometry.Morphisms.*`.

In [7]:
print(CHECKS['morphisms'])

@AlgebraicGeometry.Etale : {X Y : AlgebraicGeometry.Scheme} → (X ⟶ Y) → Prop

@AlgebraicGeometry.Smooth : {X Y : AlgebraicGeometry.Scheme} → (X ⟶ Y) → Prop

@AlgebraicGeometry.IsSeparated : {X Y : AlgebraicGeometry.Scheme} → (X ⟶ Y) → Prop


### Interpretation : proprietes locales

Les trois predicats Lean affichent une signature uniforme : `{X Y : Scheme} -> (X ⟶ Y) -> Prop`. C'est-a-dire : etant donne un morphisme `f : X ⟶ Y`, dire `Etale f`, `Smooth f`, `IsSeparated f` est une proposition.

| Propriete | Intuition |
|-----------|-----------|
| `Etale` | morphisme "plat + non ramifie" : analogue d'un revetement de surface de Riemann |
| `Smooth` | morphisme "plat + lisse au sens algebrique" : analogue d'une submersion lisse en geometrie differentielle |
| `IsSeparated` | analogue de "separe" topologique : la diagonale est fermee |

Mathlib regroupe ces proprietes sous le concept de **propriete locale** (locale sur la cible, sur la source, etc.) dans `Mathlib.AlgebraicGeometry.Morphisms.Basic`. C'est l'API qui sera utilisee pour construire le **site etale** -- la brique manquante pour parler de cohomologie etale (cf section suivante).

**Note Mathlib actuelle** : les anciens noms `IsEtale`, `IsSmooth` ont ete renommes en `Etale`, `Smooth` (depreciations visibles dans les warnings).

## 7. Ce qui est hors-scope de cet hommage

Cet hommage est volontairement court. Il y a une raison principale : **une grande partie de l'oeuvre de Grothendieck n'est pas (encore) dans Mathlib 4**, et tenter d'en parler comme si elle l'etait serait malhonnete.

### Hors scope cette serie (et probablement Mathlib 4 actuel)

| Sujet grothendieckien | Etat Mathlib 4 (mai 2026) | Pourquoi hors-scope |
|-----------------------|---------------------------|----------------------|
| Cohomologie etale ell-adique | embryonnaire (site etale pas encore complet) | tres long, requiert le site etale + faisceaux constructibles + Lefschetz |
| Motifs (cat. derivee des motifs) | absent | DM(k) requiert geometrie algebrique stable, en cours mais loin |
| Six operations (f^*, f_*, f_!, f^!, ⊗, RHom) | absent | enorme machinerie, requiert categories derivees motiviques |
| Grothendieck-Riemann-Roch (GRR) | absent | requiert K-theorie algebrique + motifs |
| Dualite de Grothendieck | absent | requiert categories derivees + categories abeliennes graduees |
| EGA II / III / IV (cohomologie schemas, faisceaux quasi-coherents profonds) | partiel, en developpement | enorme, plusieurs annees de travail Mathlib |
| Geometrie anabelienne (Tate, pi_1 etale) | absent | requiert pi_1 etale + theorie des Galois |
| Cohomologie cristalline | absent | requiert cristaux + sites cristallins |

### Pourquoi insister sur le caractere partiel

Parce que **Mathlib avance**. Joel Riou a contribue d'importants travaux sur les categories derivees en 2024-2025. Le site etale, les faisceaux quasi-coherents, l'image directe et l'image inverse progressent. Ce notebook est un instantane (mai 2026). Dans un ou deux ans, il faudra le reactualiser.

Ce qui est solide aujourd'hui : **categories, foncteurs, sites, faisceaux, schemas, site de Zariski, premieres proprietes locales**. C'est deja un programme intellectuel considerable. Le voir transcrit en Lean est, en soi, un hommage.

### Ce que cet hommage NE pretend PAS faire

1. **Pas une formalisation EGA/SGA**. Pour cela, il faudrait des annees-homme et un effort communautaire (cf Liquid Tensor Experiment, Polynomial Functional Calculus, et d'autres projets Mathlib).
2. **Pas une contribution upstream Mathlib**. Tous les `#check` montres ici existent deja dans Mathlib.
3. **Pas un cours de geometrie algebrique**. Pour cela, lire EGA, Hartshorne, Stacks Project, ou plus pedagogiquement Vakil "The Rising Sea".
4. **Pas une introduction a Lean**. Pour cela, voir Lean-1 a Lean-6 dans cette serie.

C'est un **hommage** : court, propre, qui dit "voici la trace de Grothendieck dans Mathlib, allez voir vous-meme".

## 8. Pour aller plus loin

### References historiques

1. **A. Grothendieck**, *Elements de geometrie algebrique* (avec J. Dieudonne), Publications mathematiques de l'IHES, 1960-1967 (EGA I-IV).
2. **A. Grothendieck et al.**, *Seminaire de geometrie algebrique du Bois-Marie*, plusieurs volumes, 1962-1969 (SGA 1-7).
3. **A. Grothendieck**, *Recoltes et Semailles*, manuscrit autobiographique, 1985-1986 (publie posthume, edition Gallimard 2022).
4. **A. Grothendieck**, *Tohoku paper* : "Sur quelques points d'algebre homologique", Tohoku Math. J. 9 (1957), 119-221.
5. **The Stacks Project**, https://stacks.math.columbia.edu/ : reference moderne, encyclopedique, mise a jour collaborative.
6. **R. Hartshorne**, *Algebraic Geometry*, Springer GTM 52, 1977.
7. **R. Vakil**, *The Rising Sea: Foundations of Algebraic Geometry*, draft en ligne, https://math.stanford.edu/~vakil/216blog/.

### Travaux Lean recents

- **Joel Riou** et al., travaux 2024-2025 sur les categories derivees, le foncteur derive total, les localisations de categories : cf `Mathlib.CategoryTheory.Localization.*` et `Mathlib.CategoryTheory.Triangulated.*`.
- **Kevin Buzzard**, **Adam Topaz**, **Patrick Massot** et la communaute Mathlib, ports continuels d'EGA / Stacks Project.
- **Liquid Tensor Experiment** (Scholze + Commelin et al.) : exemple recent de formalisation lourde en geometrie algebrique formelle.

### Liens vers d'autres notebooks de la serie

- [Lean-6 Mathlib Essentials](Lean-6-Mathlib-Essentials.ipynb) : tour des principales structures Mathlib
- [Lean-10 LeanDojo](Lean-10-LeanDojo.ipynb) : agents de preuve sur Mathlib
- [Lean-12 Sensitivity](Lean-12-Sensitivity-Theorem.ipynb) : un theoreme combinatoire avec preuve compacte Lean (Huang 2019)
- [Lean-15 Kochen-Specker](Lean-15-Kochen-Specker.ipynb) : meme pattern (Python kernel + subprocess WSL Lean)
- [conway_lean/](conway_lean/) : hommage Conway, modele de cet hommage Grothendieck

### Exercices courts

1. **`#check` exploratoire**. Trouver dans Mathlib les definitions de `CategoryTheory.Limits.HasLimits`, `CategoryTheory.Adjunction`, et lire leur signature. Indication : utiliser le pattern `run_lean` de ce notebook (`ALL_CHECKS` consolide pour gagner du temps).
2. **Cribles et raffinements**. Dans `Mathlib.CategoryTheory.Sites.Sieves`, identifier le lemme qui dit "raffiner un crible par un crible donne un crible" (composition de cribles).
3. **Yoneda explicite**. Lire `CategoryTheory.yonedaLemma` dans Mathlib (le lemme de Yoneda formel). Quelle est sa conclusion ?
4. **Faisceaux et recollement**. Dans `Mathlib.Topology.Sheaves.Sheaf`, identifier la condition de recollement qu'un `Presheaf` doit satisfaire pour etre un `Sheaf`. Que dit-elle intuitivement ?
5. **Pretopologie / topologie**. Dans `BigZariski.lean`, lire la preuve de `zariskiTopology_eq`. Combien de lignes ? Quelle tactique principale ?

### Note de scope (PR / Epic)

Le sous-projet `MyIA.AI.Notebooks/SymbolicAI/Lean/grothendieck_lean/` (workspace Lake avec lakefile et modules de micro-preuves) est **deliberement defere** a une Epic ulterieure (cf #1646). Cet hommage est volontairement scope-limite au notebook pour rester gerable et conforme au modele Conway tribute.

---

**Navigation** : [<< Lean-12 Sensitivity](Lean-12-Sensitivity-Theorem.ipynb) | [Index](README.md) | [Lean-15 Kochen-Specker >>](Lean-15-Kochen-Specker.ipynb)